# Historical feature and classifier comparison

Curated from the original collaborative PBL repository (source blob `47203df43ecdb278fe3c1df453a28a53dbf7e899`). Original cell order and research logic are retained; local PBL paths were made relative and saved outputs/execution metadata were cleared for privacy and size. This archival notebook documents exploratory feature/classifier experiments, not a freshly validated pipeline. Original datasets and embeddings are not included. Execute individual cells only after supplying their inputs and reviewing alternative branches. Some original cells use earlier APIs or write experiment outputs; see [data preparation notes](../../docs/data_preparation.md).


In [ ]:
import sys
sys.path.append('..')

In [ ]:
from src.preprocessing import create_dataset_from_h5, dim_reduction
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from src.utils import calculate_search_grid
from sklearn.metrics import accuracy_score, matthews_corrcoef, roc_auc_score, make_scorer

In [ ]:
train_path = Path('../data/embeddings/embeddings_uniprot_exp_proteins_c30_flens2-98_enz/embeddings_uniprot_exp_proteins_c30_flens2-98_enz_train_dim128.h5')
test_path = Path('../data/embeddings/embeddings_uniprot_exp_proteins_c30_flens2-98_enz/embeddings_uniprot_exp_proteins_c30_flens2-98_enz_test_dim128.h5')
train_set = create_dataset_from_h5(train_path, train_path.stem, unsqueeze=False)
test_set = create_dataset_from_h5(test_path, test_path.stem, unsqueeze=False)

In [ ]:
X_train = train_set.X
y_train = train_set.y
X_test = test_set.X
y_test =test_set.y

In [ ]:
# Initializing the estimators
rf = RandomForestClassifier()
logr = LogisticRegression(max_iter=1000)
svm = LinearSVC()

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import matthews_corrcoef as mcc, make_scorer

cv_rf = cross_validate(rf, X_train, y_train, scoring={'mcc': make_scorer(mcc)})
cv_logr = cross_validate(logr, X_train, y_train, scoring={'mcc': make_scorer(mcc)})
cv_svm = cross_validate(svm, X_train, y_train, scoring={'mcc': make_scorer(mcc)})

In [ ]:
rf_mean = np.mean(cv_rf['test_mcc'])
rf_std = np.std(cv_rf['test_mcc'])

logr_mean = np.mean(cv_logr['test_mcc'])
logr_std = np.std(cv_logr['test_mcc'])

svm_mean = np.mean(cv_svm['test_mcc'])
svm_std = np.std(cv_svm['test_mcc'])

dummy_mean = np.mean(cv_dummy['test_mcc'])
dummy_std = np.std(cv_dummy['test_mcc'])

model_mean = np.mean(mccs)
model_std = np.std(mccs)

In [ ]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier()
cv_dummy = cross_validate(dummy, X_train, y_train, scoring={'mcc': make_scorer(mcc)})

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data
models = ['Random\nClassifier', 'Random\nForest', 'Linear\nRegression', 'Linear\nSVM', 'Neural\nNetwork']
means = [0.00, 0.67, 0.73, 0.74, 0.80]
errors = [0, 0.038, 0.027, 0.027, 0.004] 

# 1. Define colors
colors = ["#858585"] * len(models)

# --- 2. Create Plot ---
fig, ax = plt.subplots(figsize=(10, 7))  

# Create bars
bars = ax.bar(models, means, yerr=errors, capsize=6, color=colors, alpha=1.0, width=0.6, 
              error_kw={'elinewidth': 2, 'ecolor': 'black'})

# --- 3. Styling & Formatting ---

# Y-axis settings
ax.set_ylim(0, 1.0) 
ax.set_ylabel('MCC', fontsize=18)

# CHANGED: Set y-ticks to steps of 0.2
ax.set_yticks(np.arange(0, 1.1, 0.2))

# Title
ax.set_title('Cross-Validation MCC (5 folds)', fontsize=22, pad=20)

# Tick Label Size
ax.tick_params(axis='both', which='major', labelsize=16)

# Grid
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)

# --- 4. Add Text Annotations ---
for bar, mean, error in zip(bars, means, errors):
    height = bar.get_height()
    
    if mean == 0:
        label_text = "0.00"
    else:
        label_text = f'{mean:.2f} ± {error:.3f}'
    
    ax.text(bar.get_x() + bar.get_width() / 2, 
            height + error + 0.02,   
            label_text, 
            ha='center', 
            va='bottom', 
            fontsize=16,            
            color='black')

# --- 5. Display ---
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

cv_rf = cross_validate(rf, X_train, y_train, scoring={'mcc': make_scorer(mcc)})


In [ ]:
np.mean(cv_rf['test_mcc'])

In [ ]:
from sklearn.metrics import accuracy_score, matthews_corrcoef

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
print(matthews_corrcoef(y_test, y_pred))
accuracy_score(y_test, y_pred)

In [ ]:
logr.fit(X_train, y_train)
y_pred = logr.predict(X_test)
print(matthews_corrcoef(y_test, y_pred))

In [ ]:
svm.fit(X_train, y_train)
y_pred = svm.predict(X_test)
print(matthews_corrcoef(y_test, y_pred))

In [ ]:
mcc_scorer = make_scorer(matthews_corrcoef)
roc_scorer = make_scorer(roc_auc_score)
#cv_rf = cross_validate(rf, X_train, y_train, scoring={'accuracy': 'accuracy', 'mcc': mcc_scorer, 'roc': roc_scorer})
cv_logr = cross_validate(logr, X_train, y_train, scoring={'accuracy': 'accuracy', 'mcc': mcc_scorer, 'roc': roc_scorer})
#cv_svm = cross_validate(svm, X_train, y_train, scoring={'accuracy': 'accuracy', 'mcc': mcc_scorer, 'roc': roc_scorer})

In [ ]:
# Random Forest
n_iter = 5
param_dist_rf = {'max_depth': [None, 10, 20, 30], 'max_features': ['sqrt', 'log2'], 'min_samples_split': [2, 5], 'min_samples_leaf': [1, 2, 4]}
rand_grid_rf = calculate_search_grid(rf, X_train, y_train, X_test, y_test, n_iter, param_dist_rf)
rand_grid_rf.to_csv('../results/performance_eval/gridsearch_model_comparison/prott5/prott5_rf.tsv', sep='\t', index=False)

In [ ]:
# Logistic Regression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer
n_iter = 10
param_dist_logr = {'max_iter': [1000], 'C': [0.01, 0.1, 1, 10, 100], 'tol': [0.0001, 0.001, 0.01]}
rand_grid_logr = calculate_search_grid(logr, X_train, y_train, X_test, y_test, n_iter, param_dist_logr, randomized=False)
rand_grid_logr.to_csv('../results/performance_eval/gridsearch_model_comparison/prott5/prott5_logr.tsv', sep='\t', index=False)

In [ ]:
# SVM
n_iter = 10
param_dist_svm = {'loss': ['hinge', 'squared_hinge'], 'C': [0.01, 0.1, 1, 10, 100], 'tol': [0.0001, 0.001, 0.01], 'fit_intercept': [True, False]}
rand_grid_svm = calculate_search_grid(svm, X_train, y_train, X_test, y_test, n_iter, param_dist_svm, randomized=False)
rand_grid_svm.to_csv('../results/perfomance_eval/gridsearch_model_comparison/prott5/prott5_svm.tsv', sep='\t', index=False)

In [ ]:
from sklearn.dummy import DummyClassifier

d = DummyClassifier()
d.fit(X_train, y_train)
y_pred = d.predict(X_test)
print(matthews_corrcoef(y_test, y_pred))

In [ ]:
def calculate_mcc_manual(tp, tn, fp, fn):
    numerator = (tp * tn) - (fp * fn)
    denominator = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    
    if denominator == 0:
        return 0
    return numerator / denominator

In [ ]:
results['Fold 1']

In [ ]:
mccs = list()
for i in range(1, 6):
    curr = results[f'Fold {i}']
    tp = curr['tp']
    tn = curr['tn']
    fn = curr['fn']
    fp = curr['fp']
    mccs.append(calculate_mcc_manual(tp, tn, fp, fn))


In [ ]:
from src.models import build_model_from_pretrained
import torch.nn as nn
import torch.optim as optim
from src.crossval import cross_validate
from src.infer import infer
from torch.utils.data import Subset, DataLoader
import torch

train_path = Path('../data/embeddings/embeddings_uniprot_exp_proteins_c30_flens2-98_enz/embeddings_uniprot_exp_proteins_c30_flens2-98_enz_train_dim128.h5')
test_path = Path('../data/embeddings/embeddings_uniprot_exp_proteins_c30_flens2-98_enz/embeddings_uniprot_exp_proteins_c30_flens2-98_enz_test_dim128.h5')

train_set = create_dataset_from_h5(train_path, train_path.stem, unsqueeze=False)
test_set = create_dataset_from_h5(test_path, test_path.stem, unsqueeze=False)

# model = build_model_from_pretrained('task1_model3_embeddings_uniprot_exp_proteins_c30_flens2-98_enz_train_dim128')

In [ ]:
np.std(score_list)

In [ ]:
test_scores = [0, 0.698, 0.739, 0.742, 0.82]
bootstrap_mean = [0, 0.698, 0.739, 0.743, 0.821]
bootstrap_std = [0, 0.01, 0.009, 0.009, 0.0078]

In [ ]:
rf.fit(train_set.X, train_set.y)
logr.fit(train_set.X, train_set.y)
svm.fit(train_set.X, train_set.y)

In [ ]:
print(np.mean(score_list))
print(np.std(score_list))

In [ ]:
preds = svm.predict(test_set.X)
matthews_corrcoef(test_set.y, preds)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Input Data
classifiers = ['Random\nClassifier', 'Random\nForest', 'Linear\nRegression', 'Linear\nSVM', 'Neural\nNetwork']
test_scores = [0, 0.698, 0.739, 0.742, 0.82]
bootstrap_std = [0, 0.01, 0.009, 0.009, 0.0078]

# 2. Setup Colors
colors = ['#858585'] * 5

# 3. Create Plot (Increased figure size for larger text)
fig, ax = plt.subplots(figsize=(12, 10))

# Create bars
bars = ax.bar(classifiers, test_scores, 
              yerr=bootstrap_std, 
              capsize=6, # Slightly larger caps
              color=colors, 
              width=0.6, 
              linewidth=0.5,
              error_kw={'ecolor': 'black', 'elinewidth': 2}) # Thicker error lines

# 4. Formatting Axes
# Set Y-axis strictly to 0-1 range
ax.set_ylim(0, 1.0) 
ax.set_yticks(np.arange(0, 1.2, 0.2))

# INCREASED FONT SIZES
ax.set_ylabel('MCC', fontsize=26, fontweight='bold') 
ax.set_xticklabels(classifiers, fontsize=20)
ax.tick_params(axis='y', labelsize=20)

# Grid lines
ax.yaxis.grid(True, linestyle='--', alpha=0.6, color='gray')
ax.set_axisbelow(True)

# INCREASED TITLE FONT SIZE
ax.set_title('Bootstrapped MCC Performance on Test Set', fontsize=30, pad=25)

# 5. Add Labels with ± text
for bar, score, std in zip(bars, test_scores, bootstrap_std):
    height = bar.get_height()
    
    y_pos = height + std + 0.015
    
    if score == 0:
        label_text = "0.00"
        y_pos = 0.02
    else:
        label_text = f"{score:.2f}\n±{std:.3f}"

    # INCREASED LABEL FONT SIZE
    ax.text(bar.get_x() + bar.get_width()/2.0, y_pos, 
            label_text, 
            ha='center', va='bottom', fontsize=20)

plt.tight_layout()
plt.show()

In [ ]:
score_list = list()

n = len(test_set) 

for i in range(1000):
    indices = np.random.randint(0, n, size=n)
    bootstrap_set = Subset(test_set, indices)
    loader = DataLoader(bootstrap_set, n, shuffle=False)
    for feats, labels, _ in loader:
        preds = svm.predict(feats)
        print(i)
        score_list.append(matthews_corrcoef(labels, preds))